# PMFBY: evaluate the completed pipeline
Run after notebook 01. Reads `artifacts/latest_run.json`; no hard-coded historical run is required. No generation or model fitting occurs here. New diagnostic tables use their own namespace.

Checks the loss/non-loss mix, reconciles all 18 policy/capacity combinations, exports a label-free review worklist, analyses evidence tiers and verifies score granularity. The paired bootstrap resamples context contributions to **fixed original queues**. It does not refit or rerank, does not maintain capacity in every resample, and is conditional on the simulation. Cross-season district dependence is not modelled.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = None
candidate = Path(PROJECT_ROOT or Path.cwd()).resolve()
root = (
    next((p for p in (candidate,
             *candidate.parents) if (p / 'project_config.json').is_file()),
         None)
)
if root is None:
    raise FileNotFoundError('Import/clone the COMPLETE repository; set PROJECT_ROOT if needed.')
sys.path.insert(0, str(root))
from src.project_runtime import ProjectRuntime
import json
import numpy as np
from pyspark.sql import functions as F
runtime = ProjectRuntime(spark, root, run_kind='evaluation')
run_record = json.loads((root / 'artifacts/latest_run.json').read_text())
assert run_record['status'] == 'complete'
SOURCE = run_record['prefix']
assert SOURCE.startswith(runtime.namespace + '.pmfby_pipeline_')
RUN, OUT = (runtime.run_id, runtime.prefix)
SOURCE_VERSION = 0
BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED = (2000, 20260908)
versions = {}

def read(name):
    versions[name] = SOURCE_VERSION
    return spark.read.option('versionAsOf', SOURCE_VERSION).table(SOURCE + name)
publish = runtime.publish

def rows(frame):
    return [row.asDict(recursive=True) for row in frame.collect()]


In [ ]:
labels = read('labels').select('synthetic_claim_id', 'label')
mix = rows(labels.groupBy('label').count().orderBy('label'))
assert {r['label'] for r in mix} == {0.0, 1.0}
assert sum((r['count'] for r in mix)) == runtime.config['synthetic_rows']
claims = read('synthetic_claims').filter('year = 2022')
gold = read('gold')
rankings = read('policy_rankings')
decisions = read('policy_decisions')
assert not set(decisions.columns) & {'label', 'hidden_loss_probability', 'simulated_loss_event'}
fields = (
    ['synthetic_claim_id',
         'context_key',
         'state',
         'district_censuscode',
         'year',
         'season',
         'evidence_tier',
         'feature_imputation_count',
         'weather_precip_anomaly_pct',
         'ndvi_anomaly',
         'weather_temp_max_daily_mean_c',
         'weather_valid_day_share']
)
methods = [c for c in gold.columns if c.endswith('_fill_method')]
provenance = (
    gold.select('context_key',
         F.to_json(F.struct(*[F.col(c) for c in methods])).alias('input_provenance'))
)
test_count = claims.count()
priority_budget = int(test_count * 0.2)
sample_ranks = (
    rankings.filter("policy IN ('random_forest','weather_vegetation_rule')")
    .filter(f'priority_rank <= 10 OR priority_rank BETWEEN {priority_budget - 2} AND {priority_budget
    + 2}')
)
worklist = (
    sample_ranks.join(claims.select(*fields),
         'synthetic_claim_id')
    .join(provenance,
         'context_key')
    .withColumn('review_route',
         F.when(F.col('priority_rank') <= priority_budget,
             'priority_verification').otherwise('standard_review'))
    .withColumn('human_decision_required',
         F.lit(True))
)
assert not set(worklist.columns) & {'label', 'hidden_loss_probability', 'simulated_loss_event'}
worklist = publish(worklist, 'worklist_examples')
print('WORKLIST_SAVED', OUT + 'worklist_examples', flush=True)
evaluation = (
    decisions.join(claims.select('synthetic_claim_id',
             'context_key',
             'state',
             'season',
             'evidence_tier'),
         'synthetic_claim_id')
    .join(labels,
         'synthetic_claim_id')
)
context = (
    evaluation.groupBy('context_key',
         'state',
         'season',
         'evidence_tier',
         'policy',
         'capacity_share')
    .agg(F.count('*').alias('cases'),
         F.sum('label').alias('losses'),
         F.sum(F.col('selected_for_verification').cast('long')).alias('selected_cases'),
         F.sum(F.when(F.col('selected_for_verification'),
             F.col('label')).otherwise(0.0)).alias('captured_losses'),
         F.min('score').alias('min_score'),
         F.max('score').alias('max_score'))
)
context = publish(context, 'context_capacity')
tier = (
    context.groupBy('policy',
         'capacity_share',
         'evidence_tier')
    .agg(*[F.sum(c).alias(c) for c in ['cases',
             'losses',
             'selected_cases',
             'captured_losses']])
    .withColumn('selection_rate',
         F.col('selected_cases') / F.col('cases'))
    .withColumn('within_tier_recall',
         F.when(F.col('losses') > 0,
             F.col('captured_losses') / F.col('losses')))
    .withColumn('queue_precision',
         F.when(F.col('selected_cases') > 0,
             F.col('captured_losses') / F.col('selected_cases')))
)
tier = publish(tier, 'evidence_tier_metrics')
totals = (
    rows(context.groupBy('policy',
             'capacity_share').agg(*[F.sum(c).alias(c) for c in ['cases', 'losses', 'selected_cases', 'captured_losses']]))
)
expected = rows(read('capacity_comparison'))
for actual in totals:
    ref = (
        next((r for r in expected if r['policy'] == actual['policy'] and r['capacity_share'] == actual['capacity_share']))
    )
    assert actual['cases'] == ref['candidates'] == test_count
    assert actual['losses'] == ref['total_simulated_losses']
    assert actual['selected_cases'] == ref['verification_budget']
    assert actual['captured_losses'] == ref['losses_in_priority_queue']
score_check = (
    rows(context.filter("capacity_share = 0.2 AND policy IN ('random_forest','random_forest_base_features','logistic_regression')").groupBy('policy').agg(F.count('*').alias('contexts'),
             F.sum((F.col('min_score') != F.col('max_score')).cast('long')).alias('contexts_with_different_scores')))
)
paired = (
    rows(context.filter("capacity_share = 0.2 AND policy IN ('random_forest','weather_vegetation_rule')"))
)
lookup = {}
for r in paired:
    lookup.setdefault(r['context_key'], {})[r['policy']] = r
values = []
for key in sorted(lookup):
    rf, rule = (lookup[key]['random_forest'], lookup[key]['weather_vegetation_rule'])
    assert rf['losses'] == rule['losses'] and rf['cases'] == rule['cases']
    values.append([rf['losses'], rf['captured_losses'], rule['captured_losses']])
a = np.asarray(values, dtype=float)
rng = np.random.default_rng(BOOTSTRAP_SEED)
delta = []
for _ in range(BOOTSTRAP_REPLICATES):
    total = a[rng.integers(0, len(a), size=len(a))].sum(axis=0)
    if total[0] > 0:
        delta.append(100 * (total[1] - total[2]) / total[0])
assert len(delta) == BOOTSTRAP_REPLICATES
interval = np.quantile(delta, [0.025, 0.975])
bootstrap = (
    {'replicates': BOOTSTRAP_REPLICATES,
         'seed': BOOTSTRAP_SEED,
         'contexts': len(a),
         'point_difference_pp': float(100 * (a[:, 1].sum() - a[:, 2].sum()) / a[:,
             0].sum()),
         'percentile_95_low_pp': float(interval[0]),
         'percentile_95_high_pp': float(interval[1]),
         'method': 'paired context resampling of fixed 20% queues; no retraining/reranking; conditional simulation diagnostic; district cross-season dependence not modelled'}
)
(
    publish(spark.createDataFrame([(json.dumps(bootstrap),)],
             ['diagnostic_json']),
         'bootstrap_diagnostic')
)
summary = (
    {'status': 'complete',
         'source_prefix': SOURCE,
         'source_versions': versions,
         'output_prefix': OUT,
         'full_label_mix': mix,
         'score_granularity': score_check,
         'bootstrap': bootstrap,
         'tiers_at_20_percent': rows(tier.filter('capacity_share = 0.2').orderBy('policy',
             'evidence_tier')),
         'source_acceptance': rows(read('acceptance_checks')),
         'source_stage_timings': rows(read('stage_timings')),
         'source_manifest': rows(read('run_manifest')),
         'worklist_examples': rows(worklist.orderBy('policy',
             'priority_rank')),
         'verification': 'All 18 policy/capacity totals reproduce saved source results; no source writes, new labels or model fitting.'}
)
assert all((r['passed'] for r in summary['source_acceptance']))
(
    publish(spark.createDataFrame([(json.dumps(summary, default=str),)],
             ['result_json']),
         'release_summary')
)
print('EVALUATION_COMPLETE ' + json.dumps(summary, default=str), flush=True)
display(worklist.orderBy('policy', 'priority_rank'))
